<a href="https://colab.research.google.com/github/jingwen320/skinmate_application/blob/feature%2Fmodel/skinmate_model2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
import os

# 1. Mount the Drive
drive.mount('/content/drive', force_remount=True)

# 2. Verify the file exists before unzipping
zip_path = '/content/drive/MyDrive/skinmate_dataset/Archive.zip'

if os.path.exists(zip_path):
    print("✅ Success! Archive.zip found.")
else:
    print("❌ Still can't find it. Current files in that folder:")
    # This helps us see if there is a typo in the folder name
    if os.path.exists('/content/drive/MyDrive/skinmate_dataset'):
        print(os.listdir('/content/drive/MyDrive/skinmate_dataset'))
    else:
        print("The folder 'skinmate_dataset' doesn't seem to exist in MyDrive.")

Mounted at /content/drive
✅ Success! Archive.zip found.


In [3]:
import os
from PIL import Image

data_path = '/content/skinmate_conditions_data'
bad_files = 0

print("🔍 Scanning for corrupted images...")

for root, dirs, files in os.walk(data_path):
    for file in files:
        file_path = os.path.join(root, file)
        try:
            # Try to open the image with PIL
            with Image.open(file_path) as img:
                img.verify() # Verify it's not corrupted
        except Exception:
            print(f"❌ Removing corrupted/invalid file: {file_path}")
            os.remove(file_path)
            bad_files += 1

print(f"\n✅ Cleanup finished. Removed {bad_files} bad files.")

🔍 Scanning for corrupted images...
❌ Removing corrupted/invalid file: /content/skinmate_conditions_data/__MACOSX/._dark_spots
❌ Removing corrupted/invalid file: /content/skinmate_conditions_data/__MACOSX/._pigmentation
❌ Removing corrupted/invalid file: /content/skinmate_conditions_data/__MACOSX/._wrinkles
❌ Removing corrupted/invalid file: /content/skinmate_conditions_data/__MACOSX/._redness
❌ Removing corrupted/invalid file: /content/skinmate_conditions_data/__MACOSX/._acne
❌ Removing corrupted/invalid file: /content/skinmate_conditions_data/__MACOSX/._pores
❌ Removing corrupted/invalid file: /content/skinmate_conditions_data/__MACOSX/dark_spots/._spots147.png
❌ Removing corrupted/invalid file: /content/skinmate_conditions_data/__MACOSX/dark_spots/._melasma20_jpg.rf.d7a4a452a29ee62eda4e8d21003bed83.jpg
❌ Removing corrupted/invalid file: /content/skinmate_conditions_data/__MACOSX/dark_spots/._melasma35_jpg.rf.033307ac33683dca59af977b798b6dce - Copy.jpg
❌ Removing corrupted/invalid fil

In [4]:
import shutil
import os

# Path to the unzipped folder
macosx_path = '/content/skinmate_conditions_data/__MACOSX'

if os.path.exists(macosx_path):
    shutil.rmtree(macosx_path)
    print("🧹 Successfully deleted hidden __MACOSX folder.")
else:
    print("No __MACOSX folder found. Check for other hidden folders in your class_indices.")

🧹 Successfully deleted hidden __MACOSX folder.


In [29]:
print(train_gen.class_indices)

{'dark_spots': 0, 'pores': 1, 'redness': 2, 'wrinkles': 3}


In [1]:
import zipfile
import os
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import ModelCheckpoint, ReduceLROnPlateau

# 1. Prepare Data
extract_path = '/content/skinmate_conditions_data'

if not os.path.exists(extract_path):
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(extract_path)
    print("✅ Dataset unzipped!")

# 2. Independent Data Generators
IMG_SIZE = (300, 300)
BATCH_SIZE = 32

datagen = ImageDataGenerator(
    rescale=1./255,
    validation_split=0.2,
    rotation_range=15,
    horizontal_flip=True
)

train_gen = datagen.flow_from_directory(
    extract_path, target_size=IMG_SIZE, batch_size=BATCH_SIZE,
    class_mode='categorical', subset='training'
)

val_gen = datagen.flow_from_directory(
    extract_path, target_size=IMG_SIZE, batch_size=BATCH_SIZE,
    class_mode='categorical', subset='validation'
)

# 3. Build "Fairness" Model
base_model = tf.keras.applications.EfficientNetB0(weights='imagenet', include_top=False, input_shape=(300, 300, 3))
base_model.trainable = True

x = tf.keras.layers.GlobalAveragePooling2D()(base_model.output)
x = tf.keras.layers.Dropout(0.4)(x)
# 6 Units + Sigmoid = Independent Marks
predictions = tf.keras.layers.Dense(6, activation='sigmoid')(x)

model2 = tf.keras.models.Model(inputs=base_model.input, outputs=predictions)

model2.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

# 4. Progress Saving (Model 2)
checkpoint = ModelCheckpoint(
    filepath='/content/drive/MyDrive/skinmate_conditions_best_v1.keras',
    monitor='val_loss',
    save_best_only=True,
    mode='min',
    verbose=1
)

reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, verbose=1)

# 5. Start Training
print("🚀 Training Model 2: Independent Skin Conditions...")
model2.fit(
    train_gen,
    validation_data=val_gen,
    epochs=50,
    callbacks=[checkpoint, reduce_lr]
)

Exception ignored in: <function _xla_gc_callback at 0x7ec156056160>
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/jax/_src/lib/__init__.py", line 127, in _xla_gc_callback
    def _xla_gc_callback(*args):
    
KeyboardInterrupt: 


NameError: name 'zip_path' is not defined

In [33]:
import zipfile
import os

# 1. Define the path to your zip file
zip_path = '/content/drive/MyDrive/skinmate_dataset/Archive.zip'
# 2. Define where you want to extract it (temporary Colab space is fastest)
extract_path = '/content/skinmate_data/'

# 3. Unzip
with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

print("✅ Unzip complete!")
# 4. Check the folders now
print("Folders found:", os.listdir(extract_path))

✅ Unzip complete!
Folders found: ['wrinkles', 'acne', 'pigmentation', 'redness', 'pores', '__MACOSX', 'dark_spots']
